## Extract data

In [2]:
import pandas as pd

In [5]:
import os
print(os.getcwd())

/Users/Kirill/Documents/GitHub/Data-Warehouse


In [ ]:
# # Ensure the required library is installed 
# (Can we do it (openpyxl)? Will it be a problem?)
# %pip install openpyxl

# Read the Excel file
file_path = "bitre_fatalities_dec2024.xlsx"
df = pd.read_excel(file_path, sheet_name="BITRE_Fatality", skiprows=4)
print(df.head())


   Crash ID State  Month  Year Dayweek      Time Crash Type Bus Involvement  \
0  20241115   NSW     12  2024  Friday  04:00:00     Single              No   
1  20241125   NSW     12  2024  Friday  06:15:00     Single              No   
2  20246013   Tas     12  2024  Friday  09:43:00   Multiple              No   
3  20241002   NSW     12  2024  Friday  10:35:00   Multiple              No   
4  20242261   Vic     12  2024  Friday  11:30:00   Multiple              -9   

  Heavy Rigid Truck Involvement Articulated Truck Involvement  ... Age  \
0                            No                            No  ...  74   
1                            No                            No  ...  19   
2                            No                            No  ...  33   
3                            No                            No  ...  32   
4                            -9                            -9  ...  62   

  National Remoteness Areas                           SA4 Name 2021  \
0  Inner 

In [8]:
# Clean the columnnames
# Remove leading and trailing whitespace, convert to lowercase, and replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['crash_id', 'state', 'month', 'year', 'dayweek', 'time', 'crash_type',
       'bus_involvement', 'heavy_rigid_truck_involvement',
       'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender',
       'age', 'national_remoteness_areas', 'sa4_name_2021',
       'national_lga_name_2021', 'national_road_type', 'christmas_period',
       'easter_period', 'age_group', 'day_of_week', 'time_of_day'],
      dtype='object')

In [9]:
# Add a serial number for each person killed in the accident
df['victim_number'] = df.groupby('crash_id').cumcount() + 1
print(df)



       crash_id state  month  year    dayweek      time crash_type  \
0      20241115   NSW     12  2024     Friday  04:00:00     Single   
1      20241125   NSW     12  2024     Friday  06:15:00     Single   
2      20246013   Tas     12  2024     Friday  09:43:00   Multiple   
3      20241002   NSW     12  2024     Friday  10:35:00   Multiple   
4      20242261   Vic     12  2024     Friday  11:30:00   Multiple   
...         ...   ...    ...   ...        ...       ...        ...   
56869  19896006   Tas      1  1989  Wednesday  20:20:00   Multiple   
56870  19896006   Tas      1  1989  Wednesday  20:20:00   Multiple   
56871  19896006   Tas      1  1989  Wednesday  20:20:00   Multiple   
56872  19896006   Tas      1  1989  Wednesday  20:20:00   Multiple   
56873  19895133    WA      1  1989  Wednesday  21:00:00   Multiple   

      bus_involvement heavy_rigid_truck_involvement  \
0                  No                            No   
1                  No                            

In [12]:
# Save the cleaned DataFrame to a new Excel file
output_file_path = "bitre_fatalities_cleaned.xlsx"
df.to_excel(output_file_path, index=False)
print(f"Cleaned data saved to {output_file_path}")


Cleaned data saved to bitre_fatalities_cleaned.xlsx


## Data transformation

Data transformation to apply:

1. dayweek: Drop
2. time: Categorise into rush time and usual time:
    Rush hours: 
    Morning Peak:
    Typically between 7 am and 9 am, as commuters head to work or school. 

    Evening Peak:
    Typically between 4 pm and 6 pm, as commuters travel home from work or school. 

    Not holiday, not weekend

Can be improved according to the state, city and so on

3. bus_involvement, heavy_rigid_truck_involvement, articulated_truck_involvement - treat -9 missing values
4. speed_limit: Categorise as follows:
    For all except NT:
        0-40 - low
        41-50 - med
        51-80 - high
        81 - inf - very high
    
    For NT:
        0-40 - low
        41-60 - med
        61-80 - high
        81 - inf - very high

    treat -9 as missing value
5. road_user:
    treat Other/-9, Unknown - as missing value

6. gender:
    treat -9 - as missing value

7. age: drop

8. national_remoteness_areas:
    treat Unknown - as missing value

9. sa4_name_2021:
    treat Unknown, Blank - as missing value

10. national_lga_name_2021:
    treat Unknown, Blank - as missing value

11. national_road_type:
    treat Undetermined - as missing value

12. christmas_period, easter_period:
    transform into is_holiday

13. age_group:
    treat -9 - as missing value

14. day_of_week:
    treat Unknown - as missing value

15. time_of_day:
    treat Unknown - as missing value

## Feature engineering (merging data)

you are required to utilise at least one of the following datasets: Dwelling Count Data or Population Data. You may choose to incorporate both of these additional datasets if desired.